In [ ]:
# Load data
import pandas as pd
import numpy as np
import datamol as dm
from rdkit import Chem

df_standardized = pd.read_csv('../data/standardized_ligands_sdf.csv')

In [ ]:
descriptors_df = pd.read_parquet('../data/train_descriptors.parquet')
df = df_standardized.merge(descriptors_df, on='SMILES', how='left')
df = df[['PDB', 'HET', 'Chain', 'SMILES', 'mw', 'fsp3', 'n_rotatable_bonds', 'tpsa', 'n_lipinski_hbd', 'n_lipinski_hba']].copy()

df['SMILES'] = df['SMILES'].str.split('.')
df = df.explode('SMILES')

df['Molecule'] = df['SMILES'].apply(lambda x: dm.to_mol(x) if isinstance(x, str) else None)


In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

def _preprocess(i, row):
    try:
        dm.disable_rdkit_log()
        mol = dm.to_mol(row['SMILES'], ordered=True)
        if mol is None:
            return None
        mol = dm.fix_mol(mol)
        mol = dm.sanitize_mol(mol, sanifix=True, charge_neutral=False)
        mol = dm.standardize_mol(mol, disconnect_metals=False, normalize=True, reionize=True, uncharge=True, stereo=True)
        return dm.standardize_smiles(dm.to_smiles(mol))
    except Exception:
        return None

df['standard_smiles'] = Parallel(n_jobs=-1)(
    delayed(_preprocess)(i, row) for i, row in tqdm(df.iterrows(), total=len(df))
)


In [ ]:
import pandas as pd


dups = df.duplicated(subset=['SMILES', 'HET'], keep=False).sum()

def merge_duplicates(group):
    combined_row = group.iloc[0].copy()
    
    combined_row['PDB'] = ','.join(sorted(set(group['PDB'])))
    
    for col in ['mw', 'fsp3', 'n_rotatable_bonds', 'tpsa', 'n_lipinski_hbd', 'n_lipinski_hba']:
        unique_values = group[col].unique()
        if len(unique_values) == 1:
            combined_row[col] = unique_values[0]
        else:
            combined_row[col] = ','.join(map(str, unique_values))
    
    return combined_row

df = df.groupby(['SMILES', 'HET'], as_index=False).apply(merge_duplicates)

print(f"Number of duplicates: {dups}")


In [ ]:
df_1 = df 

In [ ]:
df = df_1

In [ ]:
df = df.drop(columns=['SMILES'])
df = df.rename(columns={'standard_smiles': 'SMILES'})

In [ ]:
### Фильтр по длине SMILES ≥ 6 символов

initial_count = len(df)
df_failed_smiles_len_ge_6 = df[df['SMILES'].str.len() < 6]
df = df[df['SMILES'].str.len() >= 6]

removed_molecules = len(df_failed_smiles_len_ge_6)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with SMILES length < 6. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_smiles_len_ge_6.drop_duplicates(subset='SMILES'), mol_col='Molecule')

In [ ]:
### Фильтр: Удаление неорганических молекул (без углерода)

def is_inorganic(smiles_list):
    inorganic_elements = ['Cl', 'Cr', 'Co', 'Ca', 'Cu', 'Cs', 'Cd']
    return np.array([
        all((element not in smiles for element in inorganic_elements)) and not any(x in smiles for x in ['C', 'c'])
        for smiles in smiles_list
    ])

initial_count = len(df)
inorganic_mask = is_inorganic(df['SMILES'].values)
df_failed_inorganic = df[inorganic_mask]
df = df[~inorganic_mask]

removed_molecules = len(df_failed_inorganic)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} inorganic molecules (no carbon). ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_inorganic.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление линейных молекул, содержащих только углерод и кислород

def is_only_carbon_and_oxygen(mol):
    if mol is None:
        return False
    return all(atom.GetSymbol() in ['C', 'O', 'H'] for atom in mol.GetAtoms())

def is_linear_molecule(mol):
    if mol is None:
        return False
    return all(atom.GetDegree() <= 2 for atom in mol.GetAtoms())

def is_linear_carbon_oxygen_chain(mol):
    return is_only_carbon_and_oxygen(mol) and is_linear_molecule(mol)

initial_count = len(df)
df_failed_linear_CO = df[df['Molecule'].apply(is_linear_carbon_oxygen_chain)]
df = df[~df['Molecule'].apply(is_linear_carbon_oxygen_chain)]

removed_molecules = len(df_failed_linear_CO)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules that are linear carbon-oxygen chains. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_linear_CO.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул, содержащих только C, O, H, не имеющих циклов, ароматичности и двойных связей

def is_only_carbon_oxygen_hydrogen(mol):
    if mol is None:
        return False
    return all(atom.GetSymbol() in ['C', 'O', 'H', 'P'] for atom in mol.GetAtoms())

def has_no_cycles(mol):
    return mol.GetRingInfo().NumRings() == 0

def has_no_aromatics(mol):
    return not any(atom.GetIsAromatic() for atom in mol.GetAtoms())

def has_no_double_bonds(mol):
    return not any(bond.GetBondType() == Chem.rdchem.BondType.DOUBLE for bond in mol.GetBonds())

def is_coh_no_cycles_no_aromatics_no_double_bonds(mol):
    return (is_only_carbon_oxygen_hydrogen(mol) and 
            has_no_cycles(mol) and 
            has_no_aromatics(mol))

initial_count = len(df)
df_failed_coh_no_cycles = df[df['Molecule'].apply(is_coh_no_cycles_no_aromatics_no_double_bonds)]
df = df[~df['Molecule'].apply(is_coh_no_cycles_no_aromatics_no_double_bonds)]

removed_molecules = len(df_failed_coh_no_cycles)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with only C, O, H, no cycles, no aromatics, and no double bonds. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_coh_no_cycles.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул с линейными алифатическими цепями и без циклов

linear_aliphatic_chain_pattern = Chem.MolFromSmarts("[CH2][CH2][CH2][CH2][CH2][CH2][CH2][CH2][CH2]")

def has_linear_chain_and_no_cycles(mol):
    if mol is None:
        return False
    has_long_chain = mol.HasSubstructMatch(linear_aliphatic_chain_pattern)
    has_cycles = mol.GetRingInfo().NumRings() > 0
    return has_long_chain and not has_cycles

initial_count = len(df)
df_failed_linear_chain_no_cycles = df[df['Molecule'].apply(has_linear_chain_and_no_cycles)]
df = df[~df['Molecule'].apply(has_linear_chain_and_no_cycles)]

removed_molecules = len(df_failed_linear_chain_no_cycles)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with linear aliphatic chains and no cycles. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_linear_chain_no_cycles.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул с длинными углеродными цепями без циклов (≥ 15 атомов)

def contains_long_carbon_chain_no_cycles(mol, min_chain_length=15):
    if mol is None:
        return False
    pattern = Chem.MolFromSmarts("[C;!R]" * min_chain_length)
    if not mol.HasSubstructMatch(pattern):
        return False
    return mol.GetRingInfo().NumRings() == 0

initial_count = len(df)
df_failed_long_carbon_chain = df[df['Molecule'].apply(lambda mol: contains_long_carbon_chain_no_cycles(mol, 15))]
df = df[~df['Molecule'].apply(lambda mol: contains_long_carbon_chain_no_cycles(mol, 15))]

removed_molecules = len(df_failed_long_carbon_chain)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with carbon chains of at least 15 atoms and no cycles. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_long_carbon_chain.drop_duplicates(subset='SMILES'), mol_col='Molecule')

In [ ]:
### Фильтр по отсутствию нежелательных атомов

# Словарь с атомами
metal_atoms = {3, 4, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 37, 38, 39,
               40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
               66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 87, 88, 89, 90,
               91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103}

heavy_atoms = {21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 37, 38, 39,
               40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 55, 56, 57,
               58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73,
               74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
               90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104,
               105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118}

unwanted_atom_dict = {'metal_atoms': metal_atoms, 'heavy_atoms': heavy_atoms}

from rdkit import Chem


def contains_unwanted_atoms(smiles, unwanted_atoms="heavy_atoms"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    atom_list = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
    unwanted_atom_set = unwanted_atom_dict.get(unwanted_atoms)
    return not set(atom_list).isdisjoint(unwanted_atom_set)


initial_count = len(df)
df_failed_unwanted_atoms = df[df['SMILES'].apply(contains_unwanted_atoms)]
df = df[~df['SMILES'].apply(contains_unwanted_atoms)]

removed_molecules = len(df_failed_unwanted_atoms)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules containing unwanted atoms. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_unwanted_atoms.drop_duplicates(subset='SMILES'), mol_col='Molecule')



In [ ]:
### Фильтр: Удаление молекул с сопряжёнными двойными связями

conjugated_double_bonds_pattern = Chem.MolFromSmarts("[CH]=[CH]-[CH]=[CH]-[CH]=[CH]")

def contains_conjugated_double_bonds(mol):
    if mol is None:
        return False
    return mol.HasSubstructMatch(conjugated_double_bonds_pattern)

initial_count = len(df)
df_failed_conjugated_bonds = df[df['Molecule'].apply(contains_conjugated_double_bonds)]
df = df[~df['Molecule'].apply(contains_conjugated_double_bonds)]

removed_molecules = len(df_failed_conjugated_bonds)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with conjugated double bonds. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_conjugated_bonds.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр по молекулярной массе < 1000

initial_count = len(df)

df['mw'] = pd.to_numeric(df['mw'], errors='coerce')
df_failed_mw_lt_800 = df[df['mw'] >= 1000]
df = df[df['mw'] < 1000]

removed_molecules = len(df_failed_mw_lt_800)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with mw >= 800. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_mw_lt_800.drop_duplicates(subset='SMILES'), mol_col='Molecule')

In [ ]:
### Фильтр по числу доноров водородных связей ≤ 18

initial_count = len(df)
df['n_lipinski_hbd'] = pd.to_numeric(df['n_lipinski_hbd'], errors='coerce')

df_failed_hbd_le_18 = df[df['n_lipinski_hbd'] > 18]
df = df[df['n_lipinski_hbd'] <= 18]

removed_molecules = len(df_failed_hbd_le_18)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with hydrogen bond donors > 18. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_hbd_le_18.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул, содержащих более 4 атомов фосфора или серы

def contains_many_P_or_S(mol):
    if mol is None:
        return False
    phosphorus_count = len([atom for atom in mol.GetAtoms() if atom.GetSymbol() == 'P'])
    sulfur_count = len([atom for atom in mol.GetAtoms() if atom.GetSymbol() == 'S'])
    has_aromatic = any(atom.GetIsAromatic() for atom in mol.GetAtoms())
    return (phosphorus_count > 3 or sulfur_count > 4) or has_aromatic

initial_count = len(df)
df_failed_many_P_S = df[df['Molecule'].apply(contains_many_P_or_S)]
df = df[~df['Molecule'].apply(contains_many_P_or_S)]

removed_molecules = len(df_failed_many_P_S)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with more than 4 phosphorus or sulfur atoms or with aromatic bonds. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_many_P_S.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул с полиэфирными цепям

polyether_chain_pattern = Chem.MolFromSmarts("[CH2]O[CH2][CH2]O[CH2][CH2]O[CH2][CH2]O[CH2]")

def contains_polyether_chain(mol):
    if mol is None:
        return False
    return mol.HasSubstructMatch(polyether_chain_pattern)

initial_count = len(df)
df_failed_polyether_chain = df[df['Molecule'].apply(contains_polyether_chain)]
df = df[~df['Molecule'].apply(contains_polyether_chain)]

removed_molecules = len(df_failed_polyether_chain)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with polyether chains. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_polyether_chain.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
### Фильтр: Удаление молекул, содержащих аскорбиновую кислоту

ascorbic_acid_pattern = Chem.MolFromSmarts("O=C1O[C@H]([C@@H](O)CO)C(O)=C1O")

def contains_ascorbic_acid(mol):
    if mol is None:
        return False
    return mol.HasSubstructMatch(ascorbic_acid_pattern)

initial_count = len(df)
df_failed_ascorbic_acid = df[df['Molecule'].apply(contains_ascorbic_acid)]
df = df[~df['Molecule'].apply(contains_ascorbic_acid)]

removed_molecules = len(df_failed_ascorbic_acid)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules containing ascorbic acid. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_ascorbic_acid.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
from rdkit import Chem


inositol_pattern = Chem.MolFromSmarts("C1(O)C(O)C(O)C(O)C(O)C1O")

def contains_inositol(mol):
    if mol is None:
        return False
    return mol.HasSubstructMatch(inositol_pattern)

initial_count = len(df)
df_failed_inositol = df[df['Molecule'].apply(contains_inositol)]
df = df[~df['Molecule'].apply(contains_inositol)]

removed_molecules = len(df_failed_inositol)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with inositol. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_inositol.drop_duplicates(subset='SMILES'), mol_col='Molecule')

In [ ]:
from rdkit import Chem


pattern = Chem.MolFromSmarts("[C,O,H]1~[C,O,H]~[C,O,H]~[C,O,H]~[C,O,H]~[C,O,H]1")

def contains_one_ring_with_oxygen_only_COH(mol):
    if mol is None:
        return False

    allowed_atoms = all(atom.GetSymbol() in ['C', 'O', 'H'] for atom in mol.GetAtoms())

    ring_info = mol.GetRingInfo()
    one_ring = ring_info.NumRings() == 1
    contains_oxygen_in_ring = any(atom.GetSymbol() == 'O' for atom in mol.GetAtoms() if atom.IsInRing())
    
    return allowed_atoms and one_ring and contains_oxygen_in_ring


initial_count = len(df)
df_failed_one_ring_COH = df[df['Molecule'].apply(contains_one_ring_with_oxygen_only_COH)]
df = df[~df['Molecule'].apply(contains_one_ring_with_oxygen_only_COH)]

removed_molecules = len(df_failed_one_ring_COH)
removed_percentage = (removed_molecules / initial_count) * 100
print(f"Removed {removed_molecules} molecules with only C, O, H and one non-aromatic ring containing oxygen. ({removed_percentage:.2f}%)")

#mols2grid.display(df_failed_one_ring_COH.drop_duplicates(subset='SMILES'), mol_col='Molecule')


In [ ]:
#mols2grid.display(df.drop_duplicates(subset='SMILES'), mol_col='Molecule')

In [33]:
import pandas as pd

failed_df = pd.concat([
    df_failed_smiles_len_ge_6,
    df_failed_inorganic,
    df_failed_linear_CO,
    df_failed_coh_no_cycles,
    df_failed_linear_chain_no_cycles,
    df_failed_long_carbon_chain,
    df_failed_unwanted_atoms,
    df_failed_conjugated_bonds,
    df_failed_mw_lt_800,
    df_failed_hbd_le_18,
    df_failed_many_P_S,
    df_failed_polyether_chain,
    df_failed_ascorbic_acid,
    df_failed_inositol,
    df_failed_one_ring_COH
])

# Удаляем дубликаты, если такие есть
failed_df = failed_df.drop_duplicates(subset=['SMILES'])

# Подсчёт общего количества и процента не прошедших молекул
initial_total = len(df_standardized)
failed_total = len(failed_df)
failed_percentage = (failed_total / initial_total) * 100

failed_df = failed_df[['PDB', 'HET', 'Chain']]

print(f"Final number of molecules that failed the filters: {failed_total} out of {initial_total} ({failed_percentage:.2f}%)")
#mols2grid.display(failed_df.drop_duplicates(subset='SMILES'), mol_col='Molecule')


Final number of molecules that failed the filters: 42080 out of 306873 (13.71%)


In [37]:
failed_df

,PDB,HET,Chain
0,"3qk6,4z3w,4z3x,4z3y,4z3z,4z40",UNL,D
2,1btx,0ZX,A
11,6v7h,4D6,B
27,"6v7h,6v7i",4D6,B
35,6yzu,Q4B,A
...,...,...,...
122303,7nef,ZDC,O
122304,7nef,ZDC,I
122305,6h9v,MFU,A
122306,2lyg,1CF,A


In [39]:
import json
import pandas as pd

with open('../data/trash_ligands.json', 'r', encoding='utf-8') as file:
    trash_ligands = json.load(file)

het_values = failed_df['HET'].unique()

for het in het_values:
    if het not in trash_ligands:
        trash_ligands[het] = ""

with open('../data/trash_ligands.json', 'w', encoding='utf-8') as file:
    json.dump(trash_ligands, file, ensure_ascii=False, indent=4)

print(f"Added HET values: {het_values} to trash_ligands.json")


Added HET values: ['UNL' '0ZX' '4D6' ... 'MFU' '1CF' 'DRI'] to trash_ligands.json
